# Fine-Tuning IndoBERT untuk ABSA Hotel Santika (Kaggle)

Notebook ini melakukan **Aspect Category Sentiment Analysis (ACSA)**: untuk setiap
review, model memprediksi sentimen pada **7 aspek** sekaligus.

- **Aspek (7):** Lokasi, Kenyamanan, Pelayanan, Kebersihan, Harga, Makanan, Fasilitas
- **Kelas per aspek (4):** `none` (tidak dibahas), `positif`, `negatif`, `netral`

## Arsitektur: Multi-head shared encoder
Satu encoder **IndoBERT** dipakai bersama, lalu **7 classification head** (masing-masing
4 kelas) di atas representasi `[CLS]`. Ini efisien (1 forward pass untuk 7 aspek) dan
umum dipakai pada ACSA multi-aspek (mirip *NLI-style* / multi-task ABSA, Sun et al. 2019;
Schmitt et al. 2018 'end-to-end ACSA').

## Desain eksperimen (grid)
Kita TIDAK memakai satu konfigurasi. Berdasarkan rekomendasi literatur fine-tuning
Transformer, kita uji beberapa kombinasi hyperparameter lalu pilih yang terbaik di
**validation macro-F1**, dan laporkan **test** hanya untuk model terbaik.

| Hyperparameter | Nilai diuji | Rujukan |
|---|---|---|
| Model | `indobenchmark/indobert-base-p1`, `indolem/indobert-base-uncased` | Wilie 2020; Koto 2020 |
| Learning rate | 2e-5, 3e-5, 5e-5 | Devlin 2019 (rekomendasi BERT) |
| Batch size | 16, 32 | Devlin 2019; Mosbach 2021 |
| Epoch | 3, 4 | Devlin 2019 (2-4 epoch) |
| Max length | 128 | distribusi panjang review |
| Warmup ratio | 0.1 | Howard & Ruder 2018 (ULMFiT) |
| Weight decay | 0.01 | Loshchilov 2019 (AdamW) |
| Class weighting | on/off | Johnson 2019 (imbalanced) |

> Karena kelas **none** dominan dan beberapa aspek (Harga) minoritas, kita pakai
> **class-weighted loss** sebagai salah satu variabel dan metrik utama **macro-F1**
> (bukan accuracy) agar adil terhadap kelas minoritas (Sokolova & Lapalme 2009).

## 1. Setup environment

In [1]:
# Kaggle: aktifkan GPU di menu kanan (Settings -> Accelerator -> GPU T4 x2 atau P100).
import sys, subprocess
def pipi(pkgs):
    subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs])

# PENTING: jangan menurunkan numpy. Kaggle sekarang berbasis numpy 2.x dan
# pandas-nya dikompilasi untuk numpy 2. Memasang transformers==4.44 (butuh numpy<2)
# akan menurunkan numpy -> error 'numpy.dtype size changed' saat import pandas.
#
# Notebook ini hanya butuh transformers + accelerate (torch, pandas, numpy,
# scikit-learn sudah tersedia di Kaggle). datasets/evaluate TIDAK dipakai.
pipi(['transformers>=4.46,<5','accelerate>=0.34'])

import numpy, transformers
print('Install selesai. numpy', numpy.__version__, '| transformers', transformers.__version__)
print('Jika numpy < 2.0 di sini, lakukan: menu Run -> Restart & Clear Cell Outputs (atau Factory reset), lalu jalankan ulang dari sel ini.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Install selesai. numpy 2.4.6 | transformers 4.57.6
Jika numpy < 2.0 di sini, lakukan: menu Run -> Restart & Clear Cell Outputs (atau Factory reset), lalu jalankan ulang dari sel ini.


In [2]:
import os, json, random, gc, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModel, get_linear_schedule_with_warmup)
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE=='cuda': print('GPU:', torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 2. Load dataset (train / validation / test)

Upload 3 file hasil **Data Splitting** ke Kaggle Dataset, lalu sesuaikan `DATA_DIR`.
File: `train.csv`, `validation.csv`, `test.csv` (punya kolom 7 aspek).

Jika belum punya split, notebook ini juga bisa melakukan split sendiri dari
`dataset_absa_labeled.csv` (lihat sel fallback).

In [3]:
# Lokasi file split di Kaggle dicari OTOMATIS di dalam /kaggle/input,
# jadi tidak masalah jika dataset punya subfolder bertingkat
# (mis. /kaggle/input/absa-santika-split/absa-santika-split/train.csv).
import glob

ASPECTS = ['Lokasi','Kenyamanan','Pelayanan','Kebersihan','Harga','Makanan','Fasilitas']
LABEL2ID = {'none':0,'positif':1,'negatif':2,'netral':3}
ID2LABEL = {v:k for k,v in LABEL2ID.items()}
NUM_CLASSES = 4

def norm_label(v):
    v = str(v).strip().lower()
    return v if v in LABEL2ID else 'none'

def find_csv(filename, search_root='/kaggle/input'):
    """Cari file (mis. 'train.csv') di mana pun di bawah /kaggle/input."""
    hits = glob.glob(os.path.join(search_root, '**', filename), recursive=True)
    return hits[0] if hits else None

def load_csv(path):
    return pd.read_csv(path, encoding='utf-8-sig', dtype=str).fillna('')

tr_path = find_csv('train.csv')
va_path = find_csv('validation.csv')
te_path = find_csv('test.csv')

if tr_path and va_path and te_path:
    train_df = load_csv(tr_path)
    val_df   = load_csv(va_path)
    test_df  = load_csv(te_path)
    print('Loaded split:')
    print('  train     :', tr_path)
    print('  validation:', va_path)
    print('  test      :', te_path)
else:
    print('Split train/validation/test tidak lengkap ditemukan.')
    print('  train.csv      ->', tr_path)
    print('  validation.csv ->', va_path)
    print('  test.csv       ->', te_path)
    print('Fallback: split sendiri dari dataset_absa_labeled.csv ...')
    lab_path = find_csv('dataset_absa_labeled.csv')
    if not lab_path:
        raise FileNotFoundError('Tidak menemukan train/val/test maupun dataset_absa_labeled.csv di /kaggle/input. Pastikan dataset sudah di-Add ke notebook (panel kanan).')
    df = load_csv(lab_path)
    has = df[ASPECTS].apply(lambda r: any(str(x).strip() for x in r), axis=1)
    df = df[has].reset_index(drop=True)
    from sklearn.model_selection import train_test_split
    tv, test_df = train_test_split(df, test_size=0.10, random_state=42)
    train_df, val_df = train_test_split(tv, test_size=1/9, random_state=42)
    train_df=train_df.reset_index(drop=True); val_df=val_df.reset_index(drop=True); test_df=test_df.reset_index(drop=True)
    print('Split dibuat dari:', lab_path)

# Validasi kolom yang dibutuhkan ada
TEXT_COL = 'Text_Review' if 'Text_Review' in train_df.columns else 'text_review'
missing = [c for c in ([TEXT_COL] + ASPECTS) if c not in train_df.columns]
if missing:
    raise ValueError(f'Kolom berikut tidak ada di data: {missing}. Kolom tersedia: {list(train_df.columns)}')

print('Train:', len(train_df), '| Val:', len(val_df), '| Test:', len(test_df))

Loaded split:
  train     : /kaggle/input/datasets/vince0014/absa-santika-split/absa-santika-split/train.csv
  validation: /kaggle/input/datasets/vince0014/absa-santika-split/absa-santika-split/validation.csv
  test      : /kaggle/input/datasets/vince0014/absa-santika-split/absa-santika-split/test.csv
Train: 10706 | Val: 1267 | Test: 1309


## 3. Bangun target multi-aspek

Setiap review -> vektor 7 label (satu per aspek), tiap label di {0:none,1:positif,2:negatif,3:netral}.

In [4]:
def to_targets(df):
    Y = np.zeros((len(df), len(ASPECTS)), dtype=np.int64)
    for i, a in enumerate(ASPECTS):
        Y[:, i] = df[a].map(norm_label).map(LABEL2ID).values
    return Y

TEXT_COL = 'Text_Review' if 'Text_Review' in train_df.columns else 'text_review'
y_train = to_targets(train_df)
y_val   = to_targets(val_df)
y_test  = to_targets(test_df)
print('Bentuk target train:', y_train.shape)

# Distribusi kelas (cek imbalance)
for i,a in enumerate(ASPECTS):
    vals, cnts = np.unique(y_train[:,i], return_counts=True)
    dist = {ID2LABEL[v]:int(c) for v,c in zip(vals,cnts)}
    print(f'{a:12s}', dist)

Bentuk target train: (10706, 7)
Lokasi       {'none': 7090, 'positif': 3263, 'negatif': 205, 'netral': 148}
Kenyamanan   {'none': 6132, 'positif': 3107, 'negatif': 1259, 'netral': 208}
Pelayanan    {'none': 5781, 'positif': 3955, 'negatif': 677, 'netral': 293}
Kebersihan   {'none': 7402, 'positif': 2775, 'negatif': 470, 'netral': 59}
Harga        {'none': 9914, 'positif': 487, 'negatif': 191, 'netral': 114}
Makanan      {'none': 6523, 'positif': 3018, 'negatif': 712, 'netral': 453}
Fasilitas    {'none': 7928, 'positif': 1121, 'negatif': 1203, 'netral': 454}


## 4. Definisi Dataset & Model multi-head

In [5]:
class ABSADataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_len):
        self.texts=list(texts); self.targets=targets
        self.tok=tokenizer; self.max_len=max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc=self.tok(self.texts[idx], truncation=True, max_length=self.max_len,
                     padding='max_length', return_tensors='pt')
        item={k:v.squeeze(0) for k,v in enc.items()}
        item['labels']=torch.tensor(self.targets[idx], dtype=torch.long)
        return item

In [6]:
class MultiHeadABSA(nn.Module):
    def __init__(self, model_name, n_aspects, n_classes, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleList([nn.Linear(hidden, n_classes) for _ in range(n_aspects)])
        self.n_aspects=n_aspects; self.n_classes=n_classes
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # gunakan representasi [CLS] (token pertama)
        cls = out.last_hidden_state[:,0]
        cls = self.dropout(cls)
        logits = [head(cls) for head in self.heads]  # list of (B, n_classes)
        return torch.stack(logits, dim=1)  # (B, n_aspects, n_classes)

## 5. Class weights (opsional, untuk imbalance)

Bobot per kelas per aspek = inverse frequency, dipakai jika `use_class_weight=True`.

In [7]:
def compute_class_weights(y):
    # weights shape (n_aspects, n_classes)
    W = np.ones((len(ASPECTS), NUM_CLASSES), dtype=np.float32)
    for i in range(len(ASPECTS)):
        vals, cnts = np.unique(y[:,i], return_counts=True)
        freq = np.zeros(NUM_CLASSES); freq[vals]=cnts
        freq[freq==0]=1
        w = freq.sum()/(NUM_CLASSES*freq)
        W[i]=w
    return torch.tensor(W, dtype=torch.float32)

CLASS_W = compute_class_weights(y_train).to(DEVICE)
print('Class weights shape:', CLASS_W.shape)

Class weights shape: torch.Size([7, 4])


## 6. Fungsi training & evaluasi

In [8]:
def make_loss(use_class_weight):
    if use_class_weight:
        # CrossEntropy per aspek dengan bobot masing-masing
        losses=[nn.CrossEntropyLoss(weight=CLASS_W[i]) for i in range(len(ASPECTS))]
    else:
        losses=[nn.CrossEntropyLoss() for _ in range(len(ASPECTS))]
    def loss_fn(logits, labels):
        total=0.0
        for i in range(len(ASPECTS)):
            total=total+losses[i](logits[:,i,:], labels[:,i])
        return total/len(ASPECTS)
    return loss_fn

In [9]:
@torch.no_grad()
def evaluate(model, loader, return_preds=False):
    model.eval()
    all_logits=[]; all_labels=[]
    for batch in loader:
        labels=batch.pop('labels').to(DEVICE)
        batch={k:v.to(DEVICE) for k,v in batch.items()}
        logits=model(**{k:batch[k] for k in batch if k in ['input_ids','attention_mask','token_type_ids']})
        all_logits.append(logits.cpu()); all_labels.append(labels.cpu())
    logits=torch.cat(all_logits); labels=torch.cat(all_labels)
    preds=logits.argmax(-1).numpy(); labels=labels.numpy()
    # macro-F1 per aspek, lalu rata-rata
    f1s=[]; accs=[]
    per_aspect={}
    for i,a in enumerate(ASPECTS):
        f1=f1_score(labels[:,i], preds[:,i], average='macro', zero_division=0)
        acc=accuracy_score(labels[:,i], preds[:,i])
        f1s.append(f1); accs.append(acc); per_aspect[a]={'macro_f1':f1,'acc':acc}
    res={'macro_f1':float(np.mean(f1s)),'acc':float(np.mean(accs)),'per_aspect':per_aspect}
    if return_preds: return res, preds, labels
    return res

In [10]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

def train_one_config(cfg, train_df, val_df, verbose=True):
    set_seed(cfg.get('seed',42))
    tok=AutoTokenizer.from_pretrained(cfg['model_name'])
    tr_ds=ABSADataset(train_df[TEXT_COL].tolist(), y_train, tok, cfg['max_len'])
    va_ds=ABSADataset(val_df[TEXT_COL].tolist(),   y_val,   tok, cfg['max_len'])
    tr_ld=DataLoader(tr_ds,batch_size=cfg['batch_size'],shuffle=True)
    va_ld=DataLoader(va_ds,batch_size=cfg['batch_size'])

    model=MultiHeadABSA(cfg['model_name'], len(ASPECTS), NUM_CLASSES, cfg.get('dropout',0.1)).to(DEVICE)
    loss_fn=make_loss(cfg.get('use_class_weight',False))
    optim=AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg.get('weight_decay',0.01))
    total_steps=len(tr_ld)*cfg['epochs']
    sched=get_linear_schedule_with_warmup(optim,int(cfg.get('warmup_ratio',0.1)*total_steps),total_steps)

    best_f1=-1; best_state=None; history=[]
    for ep in range(cfg['epochs']):
        model.train()
        for batch in tr_ld:
            labels=batch.pop('labels').to(DEVICE)
            batch={k:v.to(DEVICE) for k,v in batch.items() if k in ['input_ids','attention_mask','token_type_ids']}
            logits=model(**batch)
            loss=loss_fn(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            optim.step(); sched.step(); optim.zero_grad()
        res=evaluate(model, va_ld)
        history.append(res['macro_f1'])
        if verbose: print(f"  epoch {ep+1}/{cfg['epochs']} val_macroF1={res['macro_f1']:.4f} val_acc={res['acc']:.4f}")
        if res['macro_f1']>best_f1:
            best_f1=res['macro_f1']; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
    # muat best state
    if best_state: model.load_state_dict(best_state)
    return model, tok, best_f1, history

## 7. Grid eksperimen

Untuk efisiensi waktu GPU Kaggle, kita pakai **grid terkurasi** (bukan full cartesian).
Setiap konfigurasi dievaluasi di validation; yang terbaik lanjut ke test.
Kamu bisa menambah/mengurangi entri pada list `EXPERIMENTS`.

In [11]:
INDOBERT_P1 = 'indobenchmark/indobert-base-p1'
INDOLEM     = 'indolem/indobert-base-uncased'

EXPERIMENTS = [
    # --- baseline & LR sweep (IndoBERT-p1) ---
    dict(name='p1_lr2e5_bs16_e3', model_name=INDOBERT_P1, lr=2e-5, batch_size=16, epochs=3, max_len=128, use_class_weight=False),
    dict(name='p1_lr3e5_bs16_e3', model_name=INDOBERT_P1, lr=3e-5, batch_size=16, epochs=3, max_len=128, use_class_weight=False),
    dict(name='p1_lr5e5_bs16_e3', model_name=INDOBERT_P1, lr=5e-5, batch_size=16, epochs=3, max_len=128, use_class_weight=False),
    # --- batch size & epoch ---
    dict(name='p1_lr3e5_bs32_e4', model_name=INDOBERT_P1, lr=3e-5, batch_size=32, epochs=4, max_len=128, use_class_weight=False),
    # --- class weighting (imbalance) ---
    dict(name='p1_lr3e5_bs16_e4_cw', model_name=INDOBERT_P1, lr=3e-5, batch_size=16, epochs=4, max_len=128, use_class_weight=True),
    # --- model alternatif (IndoLEM) ---
    dict(name='lem_lr3e5_bs16_e3', model_name=INDOLEM, lr=3e-5, batch_size=16, epochs=3, max_len=128, use_class_weight=False),
    dict(name='lem_lr3e5_bs16_e4_cw', model_name=INDOLEM, lr=3e-5, batch_size=16, epochs=4, max_len=128, use_class_weight=True),
]
print('Jumlah eksperimen:', len(EXPERIMENTS))

Jumlah eksperimen: 7


## 8. Jalankan semua eksperimen

In [12]:
results=[]
best_overall=None
for cfg in EXPERIMENTS:
    print('='*60); print('Eksperimen:', cfg['name'])
    model, tok, val_f1, hist = train_one_config(cfg, train_df, val_df)
    results.append({'name':cfg['name'], **{k:cfg[k] for k in ['model_name','lr','batch_size','epochs','use_class_weight']}, 'val_macro_f1':val_f1})
    print(f"  >> BEST val_macroF1 = {val_f1:.4f}")
    if best_overall is None or val_f1>best_overall['val_f1']:
        best_overall={'cfg':cfg,'val_f1':val_f1,'model':model,'tok':tok}
    else:
        del model; gc.collect(); torch.cuda.empty_cache()

res_df=pd.DataFrame(results).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
print(); print('=== RINGKASAN (urut val macro-F1) ===')
res_df

Eksperimen: p1_lr2e5_bs16_e3


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

  epoch 1/3 val_macroF1=0.5972 val_acc=0.8932
  epoch 2/3 val_macroF1=0.6267 val_acc=0.9075
  epoch 3/3 val_macroF1=0.6402 val_acc=0.9127
  >> BEST val_macroF1 = 0.6402
Eksperimen: lem_lr3e5_bs16_e4_cw


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

  epoch 1/4 val_macroF1=0.6073 val_acc=0.8463
  epoch 2/4 val_macroF1=0.6759 val_acc=0.8778
  epoch 3/4 val_macroF1=0.7036 val_acc=0.8928
  epoch 4/4 val_macroF1=0.7134 val_acc=0.8989
  >> BEST val_macroF1 = 0.7134

=== RINGKASAN (urut val macro-F1) ===


,name,model_name,lr,batch_size,epochs,use_class_weight,val_macro_f1
0,p1_lr3e5_bs16_e4_cw,indobenchmark/indobert-base-p1,0.00003,16,4,True,0.726544
1,p1_lr5e5_bs16_e3,indobenchmark/indobert-base-p1,0.00005,16,3,False,0.717370
2,lem_lr3e5_bs16_e4_cw,indolem/indobert-base-uncased,0.00003,16,4,True,0.713413
3,p1_lr3e5_bs32_e4,indobenchmark/indobert-base-p1,0.00003,32,4,False,0.685334
4,p1_lr3e5_bs16_e3,indobenchmark/indobert-base-p1,0.00003,16,3,False,0.684654
5,p1_lr2e5_bs16_e3,indobenchmark/indobert-base-p1,0.00002,16,3,False,0.672269
6,lem_lr3e5_bs16_e3,indolem/indobert-base-uncased,0.00003,16,3,False,0.640175


## 9. Evaluasi model terbaik di TEST set

In [13]:
best_cfg=best_overall['cfg']; best_model=best_overall['model']; best_tok=best_overall['tok']
print('Model terbaik:', best_cfg['name'], '| val_macroF1=', round(best_overall['val_f1'],4))

from torch.utils.data import DataLoader
te_ds=ABSADataset(test_df[TEXT_COL].tolist(), y_test, best_tok, best_cfg['max_len'])
te_ld=DataLoader(te_ds, batch_size=best_cfg['batch_size'])
test_res, preds, labels = evaluate(best_model, te_ld, return_preds=True)
print('TEST macro-F1:', round(test_res['macro_f1'],4), '| acc:', round(test_res['acc'],4))
print(); print('Per-aspek (test):')
for a,v in test_res['per_aspect'].items():
    print(f"  {a:12s} macroF1={v['macro_f1']:.4f} acc={v['acc']:.4f}")

Model terbaik: p1_lr3e5_bs16_e4_cw | val_macroF1= 0.7265
TEST macro-F1: 0.7198 | acc: 0.9125

Per-aspek (test):
  Lokasi       macroF1=0.7792 acc=0.9366
  Kenyamanan   macroF1=0.6879 acc=0.8380
  Pelayanan    macroF1=0.7405 acc=0.9129
  Kebersihan   macroF1=0.7147 acc=0.9565
  Harga        macroF1=0.6691 acc=0.9595
  Makanan      macroF1=0.7687 acc=0.9198
  Fasilitas    macroF1=0.6785 acc=0.8640


In [14]:
# Laporan klasifikasi detail per aspek
for i,a in enumerate(ASPECTS):
    print('='*50); print('Aspek:', a)
    print(classification_report(labels[:,i], preds[:,i], labels=[0,1,2,3],
          target_names=['none','positif','negatif','netral'], zero_division=0))

Aspek: Lokasi
              precision    recall  f1-score   support

        none       0.98      0.95      0.97       856
     positif       0.90      0.93      0.92       408
     negatif       0.67      0.77      0.71        26
      netral       0.42      0.68      0.52        19

    accuracy                           0.94      1309
   macro avg       0.74      0.83      0.78      1309
weighted avg       0.94      0.94      0.94      1309

Aspek: Kenyamanan
              precision    recall  f1-score   support

        none       0.93      0.85      0.89       737
     positif       0.82      0.82      0.82       389
     negatif       0.69      0.89      0.78       157
      netral       0.21      0.35      0.26        26

    accuracy                           0.84      1309
   macro avg       0.66      0.73      0.69      1309
weighted avg       0.86      0.84      0.84      1309

Aspek: Pelayanan
              precision    recall  f1-score   support

        none       0.97   

## 10. Simpan model terbaik

In [15]:
OUT='/kaggle/working/best_absa_indobert'
os.makedirs(OUT, exist_ok=True)
torch.save(best_model.state_dict(), os.path.join(OUT,'model_state.pt'))
best_tok.save_pretrained(OUT)
with open(os.path.join(OUT,'config.json'),'w') as f:
    json.dump({'model_name':best_cfg['model_name'],'aspects':ASPECTS,'label2id':LABEL2ID,
               'max_len':best_cfg['max_len'],'best_name':best_cfg['name'],
               'val_macro_f1':best_overall['val_f1'],'test':test_res}, f, indent=2, ensure_ascii=False)
res_df.to_csv(os.path.join(OUT,'experiment_results.csv'), index=False)
print('Tersimpan ke', OUT)
print(os.listdir(OUT))

Tersimpan ke /kaggle/working/best_absa_indobert
['experiment_results.csv', 'special_tokens_map.json', 'config.json', 'tokenizer.json', 'vocab.txt', 'model_state.pt', 'tokenizer_config.json']


## 11. Contoh inferensi

In [16]:
@torch.no_grad()
def predict(text):
    best_model.eval()
    enc=best_tok(text, truncation=True, max_length=best_cfg['max_len'], padding='max_length', return_tensors='pt')
    enc={k:v.to(DEVICE) for k,v in enc.items() if k in ['input_ids','attention_mask','token_type_ids']}
    logits=best_model(**enc)[0]  # (n_aspects, n_classes)
    out={}
    for i,a in enumerate(ASPECTS):
        out[a]=ID2LABEL[int(logits[i].argmax())]
    return {k:v for k,v in out.items() if v!='none'}

print(predict('kamarnya bersih dan staf ramah, tapi sarapannya kurang enak dan wifi lambat'))
print(predict('lokasi strategis dekat mall, harga terjangkau'))

{'Pelayanan': 'positif', 'Kebersihan': 'positif', 'Makanan': 'negatif', 'Fasilitas': 'negatif'}
{'Lokasi': 'positif', 'Harga': 'positif'}


## Catatan
- Metrik utama: **macro-F1** (adil untuk kelas minoritas).
- Model terbaik dipilih dari **validation**, dilaporkan di **test** (sekali pakai).
- Untuk reproduksibilitas: `seed=42` di semua eksperimen.
- Jika kehabisan waktu GPU, kurangi jumlah entri `EXPERIMENTS` atau `epochs`.